# מפגש 23 — מחברת מודרכת Audited
מקור מורה: bill_length_mm, bill_depth_mm, island; seed42. המפתח והציונים אינם ידועים לפני הרצה.

In [ ]:
assert len([3,5,9])*5==15
assert np.isclose(sum([0.80,0.85,0.75,0.90,0.70])/5,0.8) if "np" in globals() else abs(sum([0.80,0.85,0.75,0.90,0.70])/5-0.8)<1e-12

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score
df = pd.read_csv("https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv")
assert len(df)==344
numeric=["bill_length_mm","bill_depth_mm"]
categorical=["island"]
X=df[numeric+categorical].copy()
y=df["species"].copy()
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=42,stratify=y)
assert len(X_train)==258 and len(X_test)==86

## Pipeline: השלמה ותקנון מספריים, שכיח ו־OneHot של island

In [ ]:
numeric_pipe=Pipeline([("fill",SimpleImputer(strategy="median")),("scale",StandardScaler())])
categorical_pipe=Pipeline([("fill",SimpleImputer(strategy="most_frequent")),("encode",OneHotEncoder(handle_unknown="ignore",sparse_output=False))])
prepare=ColumnTransformer([("numeric",numeric_pipe,numeric),("categorical",categorical_pipe,categorical)])
pipe=Pipeline([("prepare",prepare),("knn",KNeighborsClassifier(weights="uniform",metric="euclidean"))])

## GridSearchCV: 3 מועמדים × 5 קיפולים; refit מוסיף התאמה על Train

In [ ]:
candidate_k=[3,5,9]
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
search=GridSearchCV(pipe,{"knn__n_neighbors":candidate_k},scoring="f1_macro",cv=cv,refit=True,return_train_score=True,n_jobs=1,error_score="raise")
search.fit(X_train,y_train)
table=pd.DataFrame(search.cv_results_)
assert len(table)==3
assert all(table["mean_test_score"].between(0,1))
assert search.best_params_["knn__n_neighbors"] in candidate_k
print(table[["param_knn__n_neighbors","mean_train_score","mean_test_score","std_test_score","rank_test_score"]].to_string(index=False))

## הערכה סופית: Test חיצוני אחרי בחירה בלבד

In [ ]:
baseline=DummyClassifier(strategy="most_frequent")
baseline_scores=cross_val_score(baseline,X_train,y_train,cv=cv,scoring="f1_macro")
baseline.fit(X_train,y_train)
prediction=search.predict(X_test)
dummy_prediction=baseline.predict(X_test)
assert len(prediction)==len(y_test)==86
print("Best parameter:",search.best_params_)
print("CV best macro-F1:",search.best_score_)
print("CV baseline macro-F1:",baseline_scores.mean())
print("Outer Test macro-F1:",f1_score(y_test,prediction,average="macro"))
print("Outer Test baseline macro-F1:",f1_score(y_test,dummy_prediction,average="macro"))
print("Outer Test Accuracy:",accuracy_score(y_test,prediction))

## מפת תהליך — מה מותר לדעת ומתי?Train חיצוני → חמישה קיפולי Validation בתוך GridSearchCV → בחירת K לפי macro-F1 → refit על כל Train → Test חיצוני פעם אחת. שלושה K כפול חמישה קיפולים הם 15 fits; refit מוסיף fit אחד. אין לקרוא `mean_test_score` כציון Test חיצוני.

In [ ]:
import numpy as npassert len([3,5,9])*5==15fold_example=[.8,.85,.75,.9,.7]  # מספרים מומצאים בלבדassert np.isclose(sum(fold_example)/len(fold_example),.8)assert len(fold_example)==5print("Synthetic fold mean:", np.mean(fold_example))

## בדיקת חלוקה וקיפוליםהקוד הבא משתמש רק במזהי שורות מתוך Train. בכל קיפול אין חפיפה בין תת-האימון ל־Validation, וכל שורה משמשת לאימות פעם אחת.

In [ ]:
assert len(X_train)+len(X_test)==len(X)assert set(X_train.index).isdisjoint(set(X_test.index))assert X_train.columns.tolist()==numeric+categoricalassert "species" not in X_train.columnsseen=np.zeros(len(X_train),dtype=int)fold_list=list(cv.split(X_train,y_train))assert len(fold_list)==5for inner_train,inner_val in fold_list:    assert set(inner_train).isdisjoint(set(inner_val))    assert len(inner_train)+len(inner_val)==len(X_train)    seen[inner_val]+=1assert np.all(seen==1)print("5 stratified folds verified")

## בדיקת הכנה בתוך Pipelineהחציון, השכיח, התקנון והקידוד נלמדים מחדש בכל `fit` פנימי. ההמחשה הבאה אינה משתמשת בנתוני Penguins: צירוף ערך אימות לחישוב הממוצע ישנה אותו שלא כחוק.

In [ ]:
toy_train=np.array([[0.],[2.],[4.]])toy_val=np.array([[100.]])safe=StandardScaler().fit(toy_train)leaked=StandardScaler().fit(np.vstack([toy_train,toy_val]))assert np.isclose(safe.mean_[0],2)assert np.isclose(leaked.mean_[0],26.5)assert not np.isclose(safe.mean_[0],leaked.mean_[0])assert pipe.named_steps["prepare"].transformers[0][0]=="numeric"assert pipe.named_steps["prepare"].transformers[1][0]=="categorical"

## בודקים את תוצאות GridSearchCV`rank_test_score` הוא דירוג ממוצעי אימות, לא חיזוי על Test. `std_test_score` הוא פיזור ציוני הקיפולים, לא רווח סמך. אין להסיק עליונות מובהקת מהפרש קטן בין ממוצעים.

In [ ]:
assert set(candidate_k)=={3,5,9}assert len(table)==3assert table["rank_test_score"].min()==1assert table["mean_test_score"].between(0,1).all()assert table["mean_train_score"].between(0,1).all()assert table["std_test_score"].ge(0).all()assert search.best_params_["knn__n_neighbors"] in candidate_kassert np.isclose(search.best_score_,table["mean_test_score"].max())assert search.best_estimator_ is not Noneassert "prepare" in search.best_estimator_.named_stepsassert "knn" in search.best_estimator_.named_stepsdisplay(table[["param_knn__n_neighbors","mean_train_score","mean_test_score","std_test_score","rank_test_score"]].sort_values("rank_test_score"))

## כרטיס יציאהמדוע `knn__n_neighbors` הוא היפר־פרמטר שנבחר ולא סטטיסטיקה שנלמדה מ־Train? מה הבעיה בתיקון K לאחר קריאת Test? אילו טענות איננו יכולים להסיק מהבדלי CV על מאגר Penguins המוכר?